# Clase 079 — SVM para regresión (SVR)

**SVR** invierte el objetivo de SVM: ajusta una función que se desvíe **a lo sumo ε** de cada
target, dentro de un *tubo* de tolerancia. Los errores menores a ε **no se penalizan**; solo
los puntos fuera del tubo (los vectores soporte) aportan a la pérdida. Vemos `LinearSVR`,
`SVR(kernel="rbf")`, el efecto de `epsilon`, y la robustez frente a outliers vs. OLS.

Requiere: `numpy`, `matplotlib`, `scikit-learn`.

## 1. El tubo ε en 2D

Generamos `y = 0.5·x + ruido`, entrenamos `LinearSVR(epsilon=0.5)` y dibujamos la recta con su
tubo ±ε. Los puntos **fuera** del tubo son los vectores soporte.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.svm import LinearSVR, SVR
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error

RND = 42
rng = np.random.default_rng(RND)
x = np.linspace(0, 5, 120)
y = 0.5 * x + rng.normal(0, 0.4, size=x.size)
X = x.reshape(-1, 1)

eps = 0.5
svr = LinearSVR(epsilon=eps, C=1.0, max_iter=10000, random_state=RND).fit(X, y)
pred = svr.predict(X)
fuera = np.abs(y - pred) > eps            # vectores soporte aproximados
print(f"puntos fuera del tubo (vectores soporte): {fuera.sum()} de {x.size}")

order = np.argsort(x)
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(x[~fuera], y[~fuera], s=18, c="steelblue", label="dentro del tubo")
ax.scatter(x[fuera], y[fuera], s=30, c="crimson", label="vectores soporte")
ax.plot(x[order], pred[order], "k-", label="prediccion")
ax.plot(x[order], pred[order] + eps, "k--")
ax.plot(x[order], pred[order] - eps, "k--", label="tubo +/- eps")
ax.set_title(f"LinearSVR con epsilon={eps}"); ax.legend()
plt.tight_layout(); plt.show()

## 2. Efecto de epsilon

ε **grande** → tubo ancho → menos vectores soporte → modelo más simple.
ε **chico** → ajuste más fino → más vectores soporte.

In [ ]:
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=RND)
print(f"{'epsilon':>7} | {'n_SV':>5} | {'MSE_test':>9}")
for e in [0.1, 0.5, 1.5]:
    m = LinearSVR(epsilon=e, C=1.0, max_iter=10000, random_state=RND).fit(Xtr, ytr)
    n_sv = int((np.abs(ytr - m.predict(Xtr)) > e).sum())
    mse = mean_squared_error(yte, m.predict(Xte))
    print(f"{e:>7} | {n_sv:>5} | {mse:>9.3f}")

## 3. Kernel RBF sobre datos no lineales

Con `y = sin(x) + ruido`, `LinearSVR` no puede capturar la curva; `SVR(kernel="rbf")` sí.
Escalamos con `StandardScaler` porque SVR es muy sensible a la escala.

In [ ]:
xs = np.linspace(-3, 3, 200)
ys = np.sin(xs) + rng.normal(0, 0.1, size=xs.size)
Xs = xs.reshape(-1, 1)

lin_svr = make_pipeline(StandardScaler(),
                        LinearSVR(epsilon=0.1, C=1.0, max_iter=10000, random_state=RND)).fit(Xs, ys)
rbf_svr = make_pipeline(StandardScaler(),
                        SVR(kernel="rbf", C=10, gamma="scale", epsilon=0.1)).fit(Xs, ys)
mae_lin = mean_absolute_error(ys, lin_svr.predict(Xs))
mae_rbf = mean_absolute_error(ys, rbf_svr.predict(Xs))
print(f"MAE LinearSVR: {mae_lin:.3f}")
print(f"MAE SVR-RBF  : {mae_rbf:.3f}")
assert mae_rbf < mae_lin, "el kernel RBF debe capturar mejor la no linealidad"

fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(xs, ys, s=12, c="lightgray", label="datos")
ax.plot(xs, lin_svr.predict(Xs), "b-", label="LinearSVR")
ax.plot(xs, rbf_svr.predict(Xs), "r-", lw=2, label="SVR RBF")
ax.set_title("LinearSVR vs. SVR-RBF sobre sin(x)"); ax.legend()
plt.tight_layout(); plt.show()

## 4. Robustez frente a outliers vs. OLS

La pérdida de SVR es **lineal** fuera del tubo (no cuadrática como OLS), así que es menos
sensible a outliers. Inyectamos outliers en un dataset lineal y comparamos.

In [ ]:
xo = np.linspace(0, 5, 80)
yo = 0.5 * xo + rng.normal(0, 0.3, size=xo.size)
yo[:4] += 12.0                       # 5% de outliers agresivos
Xo = xo.reshape(-1, 1)
grid = np.linspace(0, 5, 100).reshape(-1, 1)

modelos = {
    "LinearRegression": LinearRegression(),
    "Ridge": Ridge(alpha=1.0),
    "LinearSVR": LinearSVR(epsilon=0.3, C=1.0, max_iter=10000, random_state=RND),
}
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(xo, yo, s=20, c="gray", label="datos (con outliers)")
for nombre, mod in modelos.items():
    mod.fit(Xo, yo)
    ax.plot(grid.ravel(), mod.predict(grid), lw=2, label=nombre)
ax.plot(grid.ravel(), 0.5 * grid.ravel(), "k:", label="relacion real")
ax.set_ylim(-1, 14); ax.set_title("SVR es mas robusto a outliers que OLS/Ridge")
ax.legend(); plt.tight_layout(); plt.show()
print("OLS y Ridge se van hacia los outliers; LinearSVR se mantiene cerca de la recta real")

## Ejercicios

1. Generá `y = 0.5·x + ruido`, entrená `LinearSVR(epsilon=0.5)` y graficá la recta con el
   tubo ±ε, marcando los vectores soporte (puntos fuera del tubo).
2. Repetí variando `epsilon ∈ {0.1, 0.5, 1.5}` y reportá cómo cambian la cantidad de vectores
   soporte y el MSE en test.
3. Sobre `y = sin(x) + ruido`, compará `LinearSVR` vs. `SVR(kernel="rbf")` reportando MAE y
   graficando ambas curvas.
4. Inyectá ~5% de outliers en un dataset lineal y compará `LinearRegression`, `Ridge` y
   `LinearSVR`. ¿Cuál degrada menos?

## Conclusiones

- SVR ajusta dentro de un **tubo ε**: los errores menores a ε no se penalizan.
- `epsilon` controla el ancho del tubo (más ancho = menos vectores soporte, modelo más
  simple); `C` penaliza los puntos fuera del tubo.
- El **escalado** es imprescindible; sin él los hiperparámetros pierden sentido.
- SVR es más **robusto a outliers** que OLS porque su pérdida es lineal fuera del tubo.
- `SVR` con kernel es O(m²)–O(m³): para datasets grandes usá `LinearSVR` o
  `SGDRegressor(loss="epsilon_insensitive")`.

## ✅ Soluciones de los ejercicios

Soluciones trabajadas y comentadas de los ejercicios de la seccion 🧪 **Ejercicios** del README. Cada bloque es autocontenido, se ejecuta **sin internet** y en pocos segundos. Intenta resolver cada ejercicio por tu cuenta antes de mirar la solucion.

### Ejercicio 1 — Tubo ε en 2D
`LinearSVR` con ε=0.5; los puntos fuera del tubo son vectores de soporte.

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from sklearn.svm import LinearSVR
rng = np.random.default_rng(0)
xl = np.linspace(0, 5, 120); yl = 0.5 * xl + rng.normal(0, 0.4, xl.size); Xl = xl.reshape(-1, 1)
eps = 0.5
mdl = LinearSVR(epsilon=eps, C=1.0, max_iter=10000, random_state=42).fit(Xl, yl)
pred = mdl.predict(Xl); out = np.abs(yl - pred) > eps
print('vectores soporte (fuera del tubo):', int(out.sum()))
o = np.argsort(xl)
plt.figure(figsize=(6, 4)); plt.scatter(xl[~out], yl[~out], s=15, label='dentro')
plt.scatter(xl[out], yl[out], s=30, c='crimson', label='soporte')
plt.plot(xl[o], pred[o], 'k-'); plt.plot(xl[o], pred[o] + eps, 'k--'); plt.plot(xl[o], pred[o] - eps, 'k--')
plt.legend(); plt.title(f'tubo +/- {eps}'); plt.tight_layout(); plt.show()

### Ejercicio 2 — Efecto de `epsilon`
Tubo mas ancho -> menos vectores de soporte.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
Xtr, Xte, ytr, yte = train_test_split(Xl, yl, test_size=0.3, random_state=42)
for e in [0.1, 0.5, 1.5]:
    mm = LinearSVR(epsilon=e, C=1.0, max_iter=10000, random_state=42).fit(Xtr, ytr)
    sv = int((np.abs(ytr - mm.predict(Xtr)) > e).sum())
    print(f'eps={e:.1f}: SV={sv:3d}  MSE test={mean_squared_error(yte, mm.predict(Xte)):.3f}')

### Ejercicio 3 — Kernel RBF en datos no lineales
`SVR(rbf)` captura `sin(x)`; `LinearSVR` no puede.

In [ ]:
from sklearn.svm import SVR
from sklearn.metrics import mean_absolute_error
xs = np.linspace(0, 6, 200); ys = np.sin(xs) + rng.normal(0, 0.1, xs.size); Xs = xs.reshape(-1, 1)
Xtr, Xte, ytr, yte = train_test_split(Xs, ys, test_size=0.3, random_state=42)
lin = LinearSVR(max_iter=10000, random_state=42).fit(Xtr, ytr)
rbf = SVR(kernel='rbf', gamma='scale').fit(Xtr, ytr)
print('MAE LinearSVR:', round(mean_absolute_error(yte, lin.predict(Xte)), 3))
print('MAE SVR rbf  :', round(mean_absolute_error(yte, rbf.predict(Xte)), 3))
o = np.argsort(xs)
plt.figure(figsize=(6, 4)); plt.scatter(xs, ys, s=10, c='lightgray')
plt.plot(xs[o], lin.predict(Xs)[o], label='LinearSVR'); plt.plot(xs[o], rbf.predict(Xs)[o], label='SVR rbf')
plt.legend(); plt.title('rbf captura la curva'); plt.tight_layout(); plt.show()

### Ejercicio 4 — GridSearch de `SVR`
(offline: `make_regression` con < 5000 muestras, cuidando el costo cuadratico).

In [ ]:
from sklearn.datasets import make_regression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import GridSearchCV
Xr, yr = make_regression(n_samples=1500, n_features=8, noise=15, random_state=0)
pipe = make_pipeline(StandardScaler(), SVR())
grid = {'svr__C': [0.1, 1, 10], 'svr__gamma': ['scale', 0.01, 0.1]}
gs = GridSearchCV(pipe, grid, cv=3).fit(Xr, yr)
print('mejor:', gs.best_params_, '| R2 CV:', round(gs.best_score_, 3))

### Ejercicio 5 — Robustez vs OLS
Con 5% de outliers, la perdida ε-insensible de `LinearSVR` degrada menos.

In [ ]:
from sklearn.linear_model import LinearRegression, Ridge
xo = np.linspace(0, 5, 200); yo = 2 * xo + 1 + rng.normal(0, 0.3, xo.size)
n_out = int(0.05 * len(xo)); idx = rng.choice(len(xo), n_out, replace=False)
yo[idx] += rng.normal(30, 5, n_out)     # outliers verticales
Xo = xo.reshape(-1, 1)
for name, mdl in [('LinearRegression', LinearRegression()), ('Ridge', Ridge(alpha=1.0)),
                  ('LinearSVR', LinearSVR(epsilon=0.5, C=1.0, max_iter=10000, random_state=42))]:
    mdl.fit(Xo, yo)
    slope = float(np.ravel(mdl.coef_)[0])
    print(f'{name:16s} pendiente estimada={slope:.2f} (verdadera=2.0)')